# ICH Detection on Head CT -- Classification with GradCAM Interpretability

Binary intracranial hemorrhage detection from axial head CT slices using a fine-tuned ResNet18 with GradCAM-based clinical evaluation of model attention patterns.

**Dataset:** CT-ICH (82 patients, 2501 slices, 5 hemorrhage subtypes + fracture)  
**Approach:** Transfer learning with clinical preprocessing and interpretability analysis  
**Author:** Matt -- PGY-3 Diagnostic Radiology Resident, UTHealth Houston

## 1. Environment Setup and Data Loading

The CT-ICH dataset contains 82 patients with pre-rendered brain and bone window CT images. We use brain window only, as it captures the majority of hemorrhage subtypes relevant to detection.

In [ ]:
# --- Setup ---
# Upload Archive.zip to Google Drive, then mount
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/Archive.zip" -d /content/ich_data 2>/dev/null || \
  !unzip -q "/content/drive/MyDrive/Archive (2).zip" -d /content/ich_data 2>/dev/null || \
  print('Upload Archive.zip to Google Drive first')

!ls /content/ich_data

In [ ]:
# --- Imports ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Data Exploration

Before building the model, we need to understand the dataset structure and class distribution. Class imbalance is a critical consideration -- if hemorrhage is rare in the dataset, the model can achieve high accuracy by simply predicting 'no hemorrhage' on every slice.

In [ ]:
# --- Build dataframe mapping images to labels ---
data_dir = '/content/ich_data/Patients_CT'
labels = pd.read_csv('/content/ich_data/hemorrhage_diagnosis.csv')
labels['any_hemorrhage'] = 1 - labels['No_Hemorrhage']

records = []
for _, row in labels.iterrows():
    patient = f"{int(row['PatientNumber']):03d}"
    slice_num = int(row['SliceNumber'])
    img_path = f"{data_dir}/{patient}/brain/{slice_num}.jpg"
    if os.path.exists(img_path):
        records.append({
            'path': img_path,
            'any_hemorrhage': int(row['any_hemorrhage']),
            'intraventricular': int(row['Intraventricular']),
            'intraparenchymal': int(row['Intraparenchymal']),
            'subarachnoid': int(row['Subarachnoid']),
            'epidural': int(row['Epidural']),
            'subdural': int(row['Subdural']),
            'fracture': int(row['Fracture_Yes_No']),
            'patient': patient
        })
df = pd.DataFrame(records)

print(f"Total slices: {len(df)}")
print(f"Unique patients: {df['patient'].nunique()}")
print(f"\nClass distribution:")
print(f"  Hemorrhage: {df['any_hemorrhage'].sum()} ({df['any_hemorrhage'].mean()*100:.1f}%)")
print(f"  No hemorrhage: {(1-df['any_hemorrhage']).sum():.0f} ({(1-df['any_hemorrhage']).mean()*100:.1f}%)")
print(f"\nSubtype breakdown:")
for col in ['intraventricular','intraparenchymal','subarachnoid','epidural','subdural','fracture']:
    print(f"  {col}: {df[col].sum()}")

### Class Imbalance

Only 12.7% of slices contain hemorrhage. A naive model predicting 'no hemorrhage' on every slice would achieve 87.3% accuracy -- a meaningless metric. We address this with a weighted loss function that penalizes missed hemorrhages 7.5x more than false alarms, reflecting the clinical reality that a missed bleed is far more dangerous than an unnecessary stat read.

## 3. Patient-Level Train/Validation Split

We split on **patients, not slices**. Adjacent CT slices from the same patient are near-identical -- if slices from one patient appear in both train and validation sets, the model is effectively tested on data it has already seen. This data leakage inflates validation metrics and gives a false sense of generalization.

In [ ]:
patients = df['patient'].unique()
train_patients, val_patients = train_test_split(patients, test_size=0.2, random_state=42)
train_df = df[df['patient'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient'].isin(val_patients)].reset_index(drop=True)

print(f"Train: {len(train_patients)} patients, {len(train_df)} slices")
print(f"Val: {len(val_patients)} patients, {len(val_df)} slices")
print(f"\nTrain hemorrhage rate: {train_df['any_hemorrhage'].mean()*100:.1f}%")
print(f"Val hemorrhage rate: {val_df['any_hemorrhage'].mean()*100:.1f}%")

## 4. Dataset Class and Preprocessing

Each CT slice is a 650x650 grayscale JPG in brain window. We resize to 224x224 (ResNet input size), convert to 3-channel (ResNet expects RGB), and normalize using ImageNet statistics (required for pretrained weights).

**Augmentation rationale:** Horizontal flip is appropriate here because hemorrhage detection is side-agnostic -- a right-sided subdural is identical to a left-sided subdural from a detection standpoint. Slight rotation (10 degrees) accounts for imperfect patient positioning. Both augmentations are anatomically plausible and effectively increase training data without introducing artifacts.

In [ ]:
class ICHDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        label = row['any_hemorrhage']
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ICHDataset(train_df, transform=train_transform)
val_dataset = ICHDataset(val_df, transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Verify
images, labels_batch = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Labels: {labels_batch.sum():.0f} positive, {(1-labels_batch).sum():.0f} negative")

## 5. Model: Fine-Tuned ResNet18

We use ResNet18 pretrained on ImageNet. The early convolutional layers already detect universal visual features (edges, textures, contrast boundaries) that transfer to CT interpretation. We replace only the final classification layer -- from 1000 ImageNet classes to a single binary output.

The positive class weight (7.5x) is calculated from the training set class ratio, ensuring the model is penalized proportionally more for missing hemorrhage than for false alarms.

In [ ]:
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

pos_weight = torch.tensor(
    [(len(train_df) - train_df['any_hemorrhage'].sum()) / train_df['any_hemorrhage'].sum()]
).to(device)
print(f"Positive weight: {pos_weight.item():.1f}x")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## 6. Training

Standard training loop: forward pass, compute loss, backpropagate, update weights. Validation runs after each epoch using AUC (area under the ROC curve) as the primary metric -- AUC measures how well the model separates hemorrhage from normal across all possible thresholds, making it more informative than accuracy for imbalanced datasets.

In [ ]:
num_epochs = 10
best_auc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for images, labels_batch in train_loader:
        images, labels_batch = images.to(device), labels_batch.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)

    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels_batch in val_loader:
            images, labels_batch = images.to(device), labels_batch.to(device)
            outputs = torch.sigmoid(model(images).squeeze())
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())
    auc = roc_auc_score(all_labels, all_preds)

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'best_ich_model.pth')
        print(f"Epoch {epoch+1}/{num_epochs} -- Loss: {avg_loss:.4f} -- AUC: {auc:.4f} *** NEW BEST ***")
    else:
        print(f"Epoch {epoch+1}/{num_epochs} -- Loss: {avg_loss:.4f} -- AUC: {auc:.4f}")

print(f"\nBest AUC: {best_auc:.4f}")

## 7. Clinical Performance Metrics

AUC alone is insufficient for clinical evaluation. We need sensitivity (how many hemorrhages does the model catch?) and specificity (how often does it false alarm?). On night float, a false negative -- missed hemorrhage -- could delay emergent intervention. A false positive triggers an unnecessary stat read, which is far less harmful.

In [ ]:
model.load_state_dict(torch.load('best_ich_model.pth'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for idx, row in val_df.iterrows():
        img = Image.open(row['path']).convert('RGB')
        input_tensor = val_transform(img).unsqueeze(0).to(device)
        pred = torch.sigmoid(model(input_tensor)).item()
        all_preds.append(pred)
        all_labels.append(row['any_hemorrhage'])

val_results = pd.DataFrame({'true': all_labels, 'pred': all_preds, 'path': val_df['path']})
val_results['pred_label'] = (val_results['pred'] > 0.5).astype(int)

tp = ((val_results['true']==1) & (val_results['pred_label']==1)).sum()
tn = ((val_results['true']==0) & (val_results['pred_label']==0)).sum()
fn = ((val_results['true']==1) & (val_results['pred_label']==0)).sum()
fp = ((val_results['true']==0) & (val_results['pred_label']==1)).sum()

print(f"True positives: {tp}")
print(f"True negatives: {tn}")
print(f"False negatives (missed hemorrhage): {fn}")
print(f"False positives (false alarms): {fp}")
print(f"\nSensitivity: {tp/(tp+fn):.3f}")
print(f"Specificity: {tn/(tn+fp):.3f}")

## 8. GradCAM -- Where Is the Model Looking?

GradCAM generates a heatmap showing which image regions most influenced the model's prediction. This allows us to evaluate whether the model attends to clinically relevant anatomy or learns shortcuts from irrelevant features.

We examine both correctly classified hemorrhage cases and normal cases to understand the model's attention patterns across both classes.

In [ ]:
!pip install -q grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

model.load_state_dict(torch.load('best_ich_model.pth'))
model.eval()

target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

pos_df = val_df[val_df['any_hemorrhage'] == 1].head(4)
neg_df = val_df[val_df['any_hemorrhage'] == 0].head(4)
sample_df = pd.concat([pos_df, neg_df])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, (_, row) in enumerate(sample_df.iterrows()):
    img = Image.open(row['path']).convert('RGB')
    input_tensor = val_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(input_tensor)).item()
    grayscale_cam = cam(input_tensor=input_tensor)[0]
    img_resized = np.array(img.resize((224, 224))) / 255.0
    visualization = show_cam_on_image(img_resized, grayscale_cam, use_rgb=True)
    r, c = idx // 4, idx % 4
    axes[r][c].imshow(visualization)
    label = 'HEMORRHAGE' if row['any_hemorrhage'] == 1 else 'NORMAL'
    axes[r][c].set_title(f'True: {label}\nPred: {pred:.2f}', fontsize=11)
    axes[r][c].axis('off')

plt.suptitle('GradCAM: Where is the model looking?\nTop row = hemorrhage cases, Bottom row = normal', fontsize=14)
plt.tight_layout()
plt.savefig('gradcam_results.png', dpi=150, bbox_inches='tight')
plt.show()

### GradCAM on False Negatives -- Missed Hemorrhages

The most clinically important errors are false negatives -- hemorrhage cases the model failed to detect. Examining GradCAM on these cases reveals what the model struggles with and why.

In [ ]:
fn_df = val_results[(val_results['true'] == 1) & (val_results['pred_label'] == 0)]
fn_paths = fn_df['path'].tolist()
fn_preds = fn_df['pred'].tolist()

n_show = min(len(fn_paths), 4)
fig, axes = plt.subplots(2, n_show, figsize=(5*n_show, 10))
if n_show == 1:
    axes = axes.reshape(2, 1)

for idx in range(n_show):
    img = Image.open(fn_paths[idx]).convert('RGB')
    input_tensor = val_transform(img).unsqueeze(0).to(device)
    grayscale_cam = cam(input_tensor=input_tensor)[0]
    img_resized = np.array(img.resize((224, 224))) / 255.0
    visualization = show_cam_on_image(img_resized, grayscale_cam, use_rgb=True)
    axes[0][idx].imshow(visualization)
    axes[0][idx].set_title(f'MISSED -- Pred: {fn_preds[idx]:.2f}', fontsize=12, color='red')
    axes[0][idx].axis('off')
    axes[1][idx].imshow(img_resized)
    axes[1][idx].set_title('Raw image', fontsize=11)
    axes[1][idx].axis('off')

plt.suptitle('FALSE NEGATIVES: Hemorrhage cases the model missed\nTop = GradCAM attention, Bottom = raw CT for clinical review', fontsize=14)
plt.tight_layout()
plt.savefig('gradcam_false_negatives.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Clinical Interpretation

### Key Findings

**On hemorrhage cases:** The model attends broadly to regions containing hyperdense blood and produces confident predictions (0.96-1.00). However, the attention is diffuse rather than precisely localized to hemorrhage margins, suggesting the model detects overall density changes rather than specific hemorrhage morphology.

**On normal cases:** The model frequently attends to the skull base, temporal bone, and mastoid air cells -- structures entirely irrelevant to hemorrhage detection. This is consistent with shortcut learning: the model associates certain anatomic landmarks with the absence of hemorrhage rather than learning what hemorrhage itself looks like.

**On missed hemorrhages:** The model can only detect hemorrhage when it directly visualizes hyperdense blood. It cannot recognize secondary signs we use clinically to heighten suspicion -- pneumocephalus, cerebral edema, fractures, or midline shift. These findings often prompt us to look harder for subtle hemorrhage that would otherwise be missed.

**Fundamental limitation:** Per-slice classification evaluates each image in isolation. When we read a head CT, we build a mental model across the full study -- a vertex fracture prompts careful inspection for underlying epidural hemorrhage. This cross-slice clinical reasoning is absent from the current architecture and would require sequence modeling (CNN + LSTM) to approximate.

### Clinical Applicability

With sensitivity of 0.79, this model misses approximately 1 in 5 hemorrhages -- insufficient for standalone triage. However, the pipeline demonstrates the core components of clinical hemorrhage detection AI: transfer learning on medical images, class imbalance handling, patient-level validation, and interpretability analysis. Production systems (Viz.ai, Aidoc) use the same foundational approach at much larger scale with multi-institutional training data, sequence modeling, and extensive clinical validation.